# Project Puente — Model Validation & Benchmark Notebook

Validates the NLLB-200-distilled-600M model for the 5 PUENTE target languages.

**FLORES Codes:**
| Language | Code |
|---|---|
| English | `eng_Latn` |
| Tagalog | `tgl_Latn` |
| Chavacano | `cbk_Latn` |
| Cebuano | `ceb_Latn` |
| Hiligaynon | `hil_Latn` |

**Steps:**
1. Check system & GPU
2. Download model (one-time)
3. Load from local `ml_models/`
4. Test all 5 language pairs
5. Benchmark inference speed
6. English pivot routing test

In [1]:
# ============================================================
# Cell 1: Check System & GPU
# ============================================================
import torch
import sys
import os

print(f"Python:  {sys.version}")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")
else:
    print("⚠️  No GPU detected — will use CPU (slower)")

Python:  3.12.12 (main, Oct 10 2025, 08:52:57) [GCC 11.4.0]
PyTorch: 2.9.0+cpu
CUDA available: False
⚠️  No GPU detected — will use CPU (slower)


In [ ]:
# ============================================================
# Cell 2: Download Model to ml_models/ (ONE-TIME ONLY)
# ============================================================
# Run this cell ONCE to download the model.
# After download, the model lives in ml_models/ and loads offline.
# ============================================================

from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

MODEL_NAME = "facebook/nllb-200-distilled-600M"
LOCAL_PATH = "../ml_models/nllb-200-distilled-600M"

if os.path.isdir(LOCAL_PATH):
    print(f"✅ Model already exists at: {LOCAL_PATH}")
    print("   Skipping download.")
else:
    print(f"⏳ Downloading {MODEL_NAME} ...")
    print("   This will take a few minutes on first run.\n")
    
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)
    
    tokenizer.save_pretrained(LOCAL_PATH)
    model.save_pretrained(LOCAL_PATH)
    
    print(f"\n✅ Model saved to: {LOCAL_PATH}")

⏳ Downloading facebook/nllb-200-distilled-600M ...
   This will take a few minutes on first run.



/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


config.json:   0%|          | 0.00/846 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/564 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.3M [00:00<?, ?B/s]

...

The tied weights mapping and config for this model specifies to tie model.shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/2.46G [00:00<?, ?B/s]


✅ Model saved to: ../ml_models/nllb-200-distilled-600M


In [3]:
# ============================================================
# Cell 3: Load Model from Local Folder (Offline)
# ============================================================

from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

LOCAL_PATH = "../ml_models/nllb-200-distilled-600M"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Loading model from: {LOCAL_PATH}")
print(f"Device: {device}\n")

tokenizer = AutoTokenizer.from_pretrained(LOCAL_PATH, local_files_only=True)
model = AutoModelForSeq2SeqLM.from_pretrained(LOCAL_PATH, local_files_only=True).to(device)
model.eval()

params = sum(p.numel() for p in model.parameters())
print(f"✅ Model loaded — {params:,} parameters on {device.type.upper()}")

Loading model from: ../ml_models/nllb-200-distilled-600M
Device: cpu



Loading weights:   0%|          | 0/509 [00:00<?, ?it/s]

✅ Model loaded — 615,073,792 parameters on CPU


In [ ]:
# ============================================================
# Cell 4: Test Translation — All 5 PUENTE Languages
# ============================================================
import time

FLORES_MAP = {
    'en':  'eng_Latn',
    'tl':  'tgl_Latn',
    'cbk': 'cbk_Latn',
    'ceb': 'ceb_Latn',
    'hil': 'hil_Latn',
}

def translate(text, src_lang="eng_Latn", tgt_lang="cbk_Latn"):
    """Translate text using the loaded NLLB model."""
    tokenizer.src_lang = src_lang
    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=128).to(device)
    with torch.no_grad():
        translated = model.generate(
            **inputs,
            forced_bos_token_id=tokenizer.convert_tokens_to_ids(tgt_lang),
            max_new_tokens=128,
            num_beams=4,
        )
    return tokenizer.batch_decode(translated, skip_special_tokens=True)[0]

# --- PUENTE Test Cases ---
test_cases = [
    ("Hello, how are you?",     "eng_Latn", "cbk_Latn", "English → Chavacano"),
    ("Good morning",            "eng_Latn", "tgl_Latn", "English → Tagalog"),
    ("I love Zamboanga",        "eng_Latn", "ceb_Latn", "English → Cebuano"),
    ("Thank you very much",     "eng_Latn", "hil_Latn", "English → Hiligaynon"),
    ("Maayong buntag",          "ceb_Latn", "eng_Latn", "Cebuano → English"),
    ("Magandang umaga",         "tgl_Latn", "eng_Latn", "Tagalog → English"),
    ("Salamat gid",             "hil_Latn", "eng_Latn", "Hiligaynon → English"),
]

print(f"{'Pair':<25} {'Input':<25} {'Output':<30} {'Time'}")
print("=" * 95)

for text, src, tgt, label in test_cases:
    start = time.perf_counter()
    result = translate(text, src, tgt)
    elapsed = (time.perf_counter() - start) * 1000
    print(f"{label:<25} {text:<25} {result:<30} {elapsed:.0f}ms")

SyntaxError: unterminated string literal (detected at line 21) (ipython-input-3010277487.py, line 21)

In [ ]:
# ============================================================
# Cell 5: Benchmark — Inference Speed (ISO 25010 Performance)
# ============================================================
import time
import statistics

test_pairs = [
    ("How are you?",       "eng_Latn", "cbk_Latn", "EN→CBK"),
    ("Good morning",       "eng_Latn", "tgl_Latn", "EN→TL"),
    ("Thank you",          "eng_Latn", "ceb_Latn", "EN→CEB"),
    ("Welcome",            "eng_Latn", "hil_Latn", "EN→HIL"),
]

iterations = 5
all_times = []

for text, src, tgt, label in test_pairs:
    times = []
    for i in range(iterations):
        start = time.perf_counter()
        _ = translate(text, src, tgt)
        elapsed = (time.perf_counter() - start) * 1000
        times.append(elapsed)
    
    avg = statistics.mean(times)
    stdev = statistics.stdev(times) if len(times) > 1 else 0
    all_times.extend(times)
    print(f"  {label}: avg={avg:.0f}ms ± {stdev:.0f}ms  (min={min(times):.0f}, max={max(times):.0f})")

print(f"\n  Overall average: {statistics.mean(all_times):.0f}ms on {device.type.upper()}")
print(f"  Total runs: {len(all_times)}")

# ============================================================
# Cell 5b: Test English Pivot Routing
# ============================================================
print("\n--- English Pivot Test (non-EN pairs) ---")

pivot_cases = [
    ("Magandang umaga", "tgl_Latn", "cbk_Latn", "Tagalog → Chavacano (via English)"),
    ("Maayong buntag",  "ceb_Latn", "hil_Latn", "Cebuano → Hiligaynon (via English)"),
]

for text, src, tgt, label in pivot_cases:
    # Hop 1: src → English
    start = time.perf_counter()
    mid = translate(text, src, "eng_Latn")
    # Hop 2: English → target
    result = translate(mid, "eng_Latn", tgt)
    elapsed = (time.perf_counter() - start) * 1000
    
    print(f"\n  {label}")
    print(f"    Input:    {text}")
    print(f"    Pivot EN: {mid}")
    print(f"    Output:   {result}")
    print(f"    Time:     {elapsed:.0f}ms (2-hop)")